# Прогноз золото-урановых рудных узлов

Главный ноутбук пайплайна. Вся логика — в пакете `src/`, настройки — в `src/config.py`.
Запускать из корня репозитория или из папки `notebook/` (путь добавляется ниже).

In [ ]:
import sys
import tempfile
from pathlib import Path

# Корень репозитория в sys.path, чтобы импортировать пакет src при запуске из notebook/.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config
from src.data_loader import find_base_dir, load_all_layers
from src.features import build_grid, build_features
from src.model import (
    mark_presence, train_model, add_uncertainty, compute_prospectivity,
    mark_gold_zones, point_top_coverage,
)
from src.visualization import make_display_classes, plot_final, plot_uncertainty

## 1. Пути и загрузка данных

In [ ]:
BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / config.SHP_SUBDIR
OUT_DIR = BASE_DIR / config.OUT_SUBDIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / config.OUT_PNG_NAME

# Временная папка для ASCII-алиасов кириллических шейп-файлов (не результат).
TMP_ALIAS_DIR = Path(tempfile.mkdtemp(prefix="shp_aliases_"))

layers, points = load_all_layers(SHP_DIR, TMP_ALIAS_DIR)
mask = layers["mask"]
print("Данные:", BASE_DIR)
print("Слоёв-факторов:", len(layers) - 1, "| точек:", 0 if points is None else len(points))

## 2. Сетка и признаки

In [ ]:
grid, mask_union, grid_shape = build_grid(mask, config.CELL_SIZE)
grid = build_features(grid, layers)
print("Ячеек сетки:", len(grid), "| форма растра:", grid_shape)

## 3. Обучение модели (presence-background)

Боевая модель — ансамбль RF+GB с усреднением по нескольким подвыборкам фона (`BackgroundEnsemble`): честным сплит-тюнингом и проверкой на 50 независимых фолдах он показал значимый прирост среднего lift над чистым RF (+0.36, 95% ДИ [0.08, 0.65]). Positive — только реальные рудопроявления, фон — случайная выборка ячеек. Без псевдометок и geo-стабилизатора. Честная оценка качества — в `validation` (см. отдельный отчёт), здесь модель обучается на всех доступных данных.

In [ ]:
grid, positive_cells = mark_presence(grid, points)
grid, model = train_model(grid, positive_cells)

print("Реальных ячеек-точек (positive):", len(positive_cells))
print("Размер фоновой выборки:", min(config.TRAIN_N_BACKGROUND, len(grid) - len(positive_cells)))

## 4. Итоговая поверхность и золотые зоны

In [ ]:
grid = compute_prospectivity(grid, grid_shape)
grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)
grid = add_uncertainty(grid, model)
print("Ячеек в золотых зонах:", int(grid["gold_zone"].sum()))

## 5. Карта прогноза и важность признаков

In [ ]:
import pandas as pd

coverage = point_top_coverage(grid, positive_cells)
plot_final(grid, mask, points, OUT_PNG)
OUT_UNC = OUT_DIR / "uncertainty.png"
plot_uncertainty(grid, mask, OUT_UNC)

print("Сохранён прогноз:", OUT_PNG)
print("Сохранена неопределённость:", OUT_UNC)
print("Ячеек с реальными точками:", len(positive_cells))
cov_txt = round(coverage, 3) if coverage == coverage else "нет данных"
print("Доля реальных точек в верхних 15% прогноза (in-sample):", cov_txt)
print("Честная оценка обобщения (lift на held-out) — в notebook/validation_report.ipynb")

if model is not None:
    imp = pd.Series(model.feature_importances_, index=config.FEATURE_COLS).sort_values(ascending=False)
    print("\nВажность признаков:")
    display(imp.round(3))